# Milestone 1 — scRNA-seq receptor heatmaps (WMB-10Xv3)

Mean log2(CPM+1) per **cell type** × **brain area**, driven entirely by `query_config.yaml`.

Requires internet on first run for `AbcProjectCache` downloads.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import pandas as pd

from src.config import (
    DEFAULT_OUTPUT_DIR,
    load_config,
    start_run,
    get_parquet_path,
    restrict_config_to_genes,
)
from src.data_loaders import (
    get_abc_cache,
    load_scrna_cell_metadata,
    check_scrna_genes_in_metadata,
    load_expression_subset,
    aggregate_scrna_expression,
    family_gene_region_matrix,
    combined_heatmap_matrix,
)
from src.utils import build_brain_area_mapping
from src.plotting import plot_family_heatmap, plot_combined_heatmap

In [ ]:
# Exploration root (outside repo). Each run gets a timestamped subfolder + run_manifest.json.
EXPLORATION_ROOT = DEFAULT_OUTPUT_DIR

CONFIG_PATH = PROJECT_ROOT / "query_config.yaml"
config = load_config(CONFIG_PATH)
OUTPUT_DIR = start_run(
    PROJECT_ROOT,
    config,
    dataset="WMB-10Xv3",
    exploration_root=EXPLORATION_ROOT,
    notebook="01_scrna_heatmaps",
)
assert not str(OUTPUT_DIR).startswith(str(PROJECT_ROOT)), (
    f"OUTPUT_DIR must be outside the repo; got {OUTPUT_DIR}"
)

print(f"Genes: {len(config['_all_genes'])}")
print(f"Brain areas: {config['brain_areas']}")
print(f"Cell type level: {config['cell_type_level']}")
filt = config.get("cell_type_name_filter") or []
print(f"Cell type name filter: {filt if filt else '(none — all types)'}")
print(f"Run dir: {OUTPUT_DIR}")
print(f"Manifest: {OUTPUT_DIR / 'run_manifest.json'}")

In [ ]:
cache = get_abc_cache(config)
print("Manifest:", cache.current_manifest)

In [ ]:
cell_meta = load_scrna_cell_metadata(cache, config)
print(f"Filtered cells: {len(cell_meta):,}")
print("\nCells per brain_area (scRNA assignable label):")
print(cell_meta.groupby("brain_area").size().sort_values(ascending=False))

if config.get("_scrna_pools"):
    print("\nPooled dissection ROIs (multiple config areas -> one scRNA label):")
    for dissection, areas in config["_scrna_pools"].items():
        n = (cell_meta["brain_area"] == dissection).sum()
        print(f"  {dissection}: {n:,} cells  <- pools {areas}")

area_to_rois, _, scrna_pools = build_brain_area_mapping(cache, config["brain_areas"])
print("\nROI mapping (config area -> dissection ROI):")
for ba, rois in area_to_rois.items():
    print(f"  {ba}: {sorted(rois)[:8]}{'...' if len(rois) > 8 else ''}")

In [ ]:
requested = list(config["_all_genes"])
genes_flat_orig = dict(config["_genes_flat"])
found_in_meta, missing_meta = check_scrna_genes_in_metadata(cache, requested)

if missing_meta:
    print(f"Genes not in WMB-10X metadata — skipped ({len(missing_meta)}):")
    for gene in missing_meta:
        print(f"  {gene} ({genes_flat_orig.get(gene, 'unknown')})")
else:
    print(f"All {len(requested)} requested genes found in WMB-10X metadata.")

if not found_in_meta:
    raise RuntimeError("No requested genes found in WMB-10X metadata.")

adata = load_expression_subset(cache, found_in_meta, cell_meta, config)
if adata is None:
    raise RuntimeError("No expression data loaded; check cache downloads.")

loaded_genes = adata.var["gene_symbol"].tolist()
restrict_config_to_genes(config, loaded_genes)

missing_matrix = sorted(set(found_in_meta) - set(loaded_genes))
if missing_matrix:
    print(f"\nGenes in metadata but not in expression matrix — skipped ({len(missing_matrix)}):")
    for gene in missing_matrix:
        print(f"  {gene} ({genes_flat_orig.get(gene, 'unknown')})")

skipped = sorted(set(requested) - set(config["_all_genes"]))
print(f"\nProceeding with {len(config['_all_genes'])} / {len(requested)} genes.")
if skipped:
    print(f"Skipped: {skipped}")
print(f"Families with data: {config['_families']}")
print(adata)

In [ ]:
agg_long = aggregate_scrna_expression(adata, cell_meta, config)
print(agg_long.head())
print(f"\nAggregated rows: {len(agg_long):,}")

In [ ]:
print(f"Saving figures to {OUTPUT_DIR}")

for family in config["_families"]:
    mat = family_gene_region_matrix(agg_long, family, config)
    if mat.empty:
        warnings.warn(f"No data for family {family!r}; skipping heatmap.")
        continue
    path = plot_family_heatmap(family, mat, config, output_dir=OUTPUT_DIR)
    print(f"Saved {path}")

In [ ]:
# Combined: all cell types (optional name filter) × all receptors (family order)
combined = combined_heatmap_matrix(agg_long, config)
print(f"Combined heatmap: {combined.shape[0]} cell types × {combined.shape[1]} genes")
path = plot_combined_heatmap(combined, config, output_dir=OUTPUT_DIR)
print(f"Saved {path}")

In [ ]:
if config["output"].get("save_processed_data", True):
    parquet_path = get_parquet_path(config, output_dir=OUTPUT_DIR)
    agg_long.to_parquet(parquet_path, index=False)
    print(f"Saved aggregated matrix to {parquet_path}")

---
## Dev / smoke test (2 genes × 2 regions)

Run this cell only to validate the pipeline with a small download footprint.

In [ ]:
# # Override config for quick test
# test_config = load_config(CONFIG_PATH)
# test_config["brain_areas"] = ["STR", "TH"]
# test_config["receptors"] = {"dopamine": ["Drd1", "Drd2"]}
# genes_map = {}
# for fam, glist in test_config["receptors"].items():
#     for g in glist:
#         genes_map[g] = fam
# test_config["_genes_flat"] = genes_map
# test_config["_all_genes"] = list(genes_map)
# test_config["_families"] = list(test_config["receptors"].keys())

# test_cache = get_abc_cache(test_config)
# test_meta = load_scrna_cell_metadata(test_cache, test_config)
# print(f"Test cells: {len(test_meta):,}")
# test_adata = load_expression_subset(test_cache, test_config["_all_genes"], test_meta, test_config)
# test_agg = aggregate_scrna_expression(test_adata, test_meta, test_config)
# test_mat = family_gene_region_matrix(test_agg, "dopamine", test_config)
# plot_family_heatmap("dopamine", test_mat, test_config, output_dir=OUTPUT_DIR)
# print(f"Test heatmap saved to {OUTPUT_DIR / 'heatmap_dopamine.png'}")